# Limma解析

【エクスポートデータ】 ・DEP_list.csv：有意差基準を満たした変動タンパク質のリスト
。 ・result_labels.csv：全タンパク質の統計解析結果テーブル
。 ・target_labels.csv：ROC分析等で注目した特定マーカーのラベル情報
。

# Import Library

In [ ]:
# install.packages("tidyverse")

In [ ]:
library("tidyverse")
options(repr.plot.width=5, repr.plot.height=5)#Default=7x7

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")


for (package in c("DEqMS","WGCNA")) {
    BiocManager::install(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}
#CRAN
for (package in c("RColorBrewer", "ggrepel")) {
    install.packages(package)
    eval(bquote(library(.(package))))
    print(str_c(package, ": ", as.character(packageVersion(package))))
}

# Loading Data

In [ ]:
fileDir <- "./Data/"
fileName <- "pro_data_common_dis.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_dis <- temp
dim(temp)

fileDir <- "./Data/"
fileName <- "pro_sample_dis.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
colnames(temp)[1] <- "SampleID"
head(temp)
pro_sample_dis <- temp


fileDir <- "./Data/"
fileName <- "pro_data_common_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data_val <- temp
dim(temp)

fileDir <- "./Data/"
fileName <- "pro_sample_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path)
colnames(temp)[1] <- "SampleID"
head(temp)
pro_sample_val <- temp


In [ ]:
pro_data_val_ori <- pro_data_val

In [ ]:
map <- read.csv("./Data/map_OID_Uni_Sym.csv", row.names = 1)

# UniProtをキーにしてSymbolを取得
uni_to_symbol <- setNames(map$Symbol, map$UniProt)

temp <- pro_data_val
# pro_data_valの列名をSymbolに変換
colnames(temp) <- ifelse(
  colnames(temp) %in% names(uni_to_symbol),
  uni_to_symbol[colnames(temp)],
  colnames(temp)
)
head(temp)
pro_data_val <- temp
head(pro_data_val)

## 重複したIDを確認

In [ ]:
duplicated_names <- colnames(pro_data_val)[duplicated(colnames(pro_data_val))]

which(colnames(pro_data_val) == "PALM2AKAP2")
pro_data_val[, which(colnames(pro_data_val) == "PALM2AKAP2")]

In [ ]:
# Symbolが重複しているものを抽出
map_dup <- map[
  duplicated(map$Symbol) | duplicated(map$Symbol, fromLast = TRUE),
  c("OlinkID", "UniProt", "Symbol")
]

# Symbolごとに並べる
map_dup <- map_dup[order(map_dup$Symbol), ]

map_dup

In [ ]:
# UniProt → Symbol
uni_to_symbol <- setNames(map$Symbol, map$UniProt)

# pro_data_valのUniProt列名を取得
uniprot_names <- colnames(pro_data_val_ori)

# Symbolに変換
symbol_names <- ifelse(
  uniprot_names %in% names(uni_to_symbol),
  uni_to_symbol[uniprot_names],
  uniprot_names
)

# 重複しているSymbolを確認
dup_symbols <- unique(symbol_names[duplicated(symbol_names)])

# 重複しているSymbolだけ、Symbol(UniProt)にする
for (i in seq_along(symbol_names)) {
  if (symbol_names[i] %in% dup_symbols) {
    symbol_names[i] <- paste0(
      symbol_names[i],
      "(",
      uniprot_names[i],
      ")"
    )
  }
}

temp <- pro_data_val
# 列名を変更
colnames(temp) <- symbol_names
head(temp)
colnames(temp)[grepl("^PALM2AKAP2", colnames(pro_data_val))]
pro_data_val <- temp

In [ ]:
write.csv(pro_data_val, file = "Data/pro_data_common_val_Assay.csv", row.names = TRUE)

# Design Matrix

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

for (sample in rownames(pro_data_val)) {
    # group <- pro_sample_val[rownames(pro_data_val) %in% pro_sample_val$SampleID,]$progress_group
    matched_row <- pro_sample_val[pro_sample_val$SampleID == sample, ]
    
    group <- if(nrow(matched_row) > 0) matched_row$progress_group else NA
    sample_group <- c(sample_group, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$sex else NA
    sample_gender <- c(sample_gender, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$fvage else NA
    sample_age <- c(sample_age, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$bmi else NA
    sample_bmi <- c(sample_bmi, group)
    
    # print(sample_group)
    # table(rownames(pro_data_val) %in% pro_sample_val$SampleID)
}
group_order <- c(0, 1, 2)
sample_group <- factor(sample_group, levels = group_order, labels = c("Control", "Early_Stage", "Advanced_Stage"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Control", "Early_Stage", "Advanced_Stage", "Female", "Age", "BMI")
design

## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_val
# print(tempD)
# str(tempD2$progress_group)

tempD2 %>%
    dplyr::mutate(Group = factor(progress_group, levels = c(0, 1, 2), labels = c("Control", "Early_Stage", "Advanced_Stage"))) %>%
    # dplyr::mutate(Group = as.character(progress_group)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_val))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_val_t <- t(pro_data_val)


fit <- lmFit(pro_data_val_t, design)
contrast <- makeContrasts(
    Early_vs_Control = Early_Stage - Control,
    Advanced_vs_Control = Advanced_Stage - Control,
    Advanced_vs_Early = Advanced_Stage - Early_Stage,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)

#Early_vs_Control
print("Early_vs_Control")
result_Early_vs_Control <- topTable(
    fit2,
    coef = "Early_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Early_vs_Control)

sig_raw <- sum(result_Early_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Early_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Early_vs_Control$adj.P.Val < 0.05 &
    abs(result_Early_vs_Control$logFC) > 0.5
)


#Advanced_vs_Control
print("Advanced_vs_Control")
result_Advanced_vs_Control <- topTable(
    fit2,
    coef = "Advanced_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Control)

sig_raw <- sum(result_Advanced_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Control$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Control$logFC) > 0.5
)

#Advanced_vs_Early
print("Advanced_vs_Early")
result_Advanced_vs_Early <- topTable(
    fit2,
    coef = "Advanced_vs_Early",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Early)

sig_raw <- sum(result_Advanced_vs_Early$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Early$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Early$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Early$logFC) > 0.5
)

# Export result data

In [ ]:
contrast_names <- colnames(contrast)

DEPlist <- lapply(contrast_names, function(coef_name) {
  
  result <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  result %>%
    mutate(
      DEP = case_when(
        adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
        adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
        TRUE ~ "NS"
      )
    ) %>%
    filter(DEP != "NS") %>%
    arrange(adj.P.Val)
})

names(DEPlist) <- contrast_names
head(DEPlist)

DEP_combined <- bind_rows(
  lapply(names(DEPlist), function(name) {
    DEPlist[[name]] %>%
      mutate(Protein = rownames(DEPlist[[name]]),　Contrast = name)%>%
      relocate(Protein)
  })
)

write.csv(
  DEP_combined,
  file = "./Data/DEP_list.csv",
  row.names = FALSE
)

In [ ]:
library(ggrepel)

contrast_names <- colnames(contrast)

volcano_plots <- lapply(contrast_names, function(coef_name) {
  
  # そのcontrastの結果を取得
  result_table <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  # DEP分類
  plot_data <- result_table %>%
    mutate(
      threshold = case_when(
        adj.P.Val < 0.05 & logFC > 0.5  ~ "Up",
        adj.P.Val < 0.05 & logFC < -0.5 ~ "Down",
        TRUE ~ "NS"
      )
    )
  
  # ラベルするタンパク質
  label_data <- bind_rows(
    plot_data %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(adj.P.Val) %>%
      head(10),
    
    plot_data %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(desc(abs(logFC))) %>%
      head(10)
  ) %>%
    distinct()
  
  # Volcano plot
  p <- ggplot(
    plot_data,
    aes(x = logFC, y = -log10(P.Value), color = threshold)
  ) +
    geom_point(alpha = 1, size = 1.5) +
    
    scale_color_manual(
      values = c(
        "Up" = "#E41A1C",
        "Down" = "#377EB8",
        "NS" = "grey"
      )
    ) +
    
    geom_text_repel(
      data = label_data,
      aes(label = rownames(label_data)),
      size = 5,
      box.padding = 0.5,
      max.overlaps = Inf
    ) +
    
    geom_vline(
      xintercept = c(-0.5, 0.5),
      linetype = "dashed",
      color = "black",
      alpha = 0.5
    ) +
    
    geom_hline(
      yintercept = -log10(0.05),
      linetype = "dashed",
      color = "black",
      alpha = 0.5
    ) +
    
    theme_minimal() +
    
    labs(
      title = paste0("Volcano Plot: ", coef_name),
      subtitle = "Adjusted for Sex, Age, and BMI",
      x = "log2 Fold Change",
      y = "-log10 P-value"
    ) +
    
    theme(legend.position = "none")
  
  return(p)
})

# 名前を付ける
names(volcano_plots) <- contrast_names

volcano_plots$Early_vs_Control
volcano_plots$Advanced_vs_Control
volcano_plots$Advanced_vs_Early

In [ ]:

contrast_names <- colnames(contrast)

boxplots <- lapply(contrast_names, function(coef_name) {
  
  # -----------------------------
  # 1. contrastの結果を取得
  # -----------------------------
  result_table <- topTable(
    fit2,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH"
  )
  
  # -----------------------------
  # 2. ラベルするタンパク質を選択
  #    adjusted P-value 上位7
  #    |logFC| 上位7
  # -----------------------------
  label_data <- bind_rows(
    result_table %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(adj.P.Val) %>%
      head(10),
    
    result_table %>%
      filter(adj.P.Val < 0.05) %>%
      arrange(desc(abs(logFC))) %>%
      head(10)
  ) %>%
    distinct()
  
  target_proteins <- rownames(label_data)
  
  # -----------------------------
  # 3. validation dataから取得
  # -----------------------------
  plot_data_long <- pro_data_val[, target_proteins, drop = FALSE] %>%
    as.data.frame() %>%
    mutate(
      Group = factor(
        pro_sample_val$progress_group,
        levels = c(0, 1, 2),
        labels = c(
          "Control",
          "Early-Stage",
          "Advanced-Stage"
        )
      )
    ) %>%
    pivot_longer(
      cols = -Group,
      names_to = "Protein",
      values_to = "Expression"
    )
    plot_data_long$Protein <- factor(
        plot_data_long$Protein,
        levels = target_proteins
  )
  
  # -----------------------------
  # 4. boxplot
  # -----------------------------
  p <- ggplot(
    plot_data_long,
    aes(x = Group, y = Expression, fill = Group)
  ) +
    geom_boxplot(
      outlier.shape = NA,
      alpha = 1,
      width = 0.6
    ) +
    # geom_jitter(
    #   width = 0.2,
    #   alpha = 0.7,
    #   size = 0.7,
    #   color = "black"
    # ) +
    facet_wrap(
      ~ Protein,
      scales = "free_y",
      ncol = 4
    ) +
    scale_fill_manual(
      values = c(
        "Control" = "royalblue",
        "Early-Stage" = "orange",
        "Advanced-Stage" = "tomato"
      )
    ) +
    theme_bw() +
    labs(
      title = paste0(
        "Top Significant Proteins: ",
        coef_name
      ),
      x = NULL,
      y = "NPX Value"
    ) +
    theme(
      legend.position = "none",
      strip.text = element_text(face = "bold"),
      axis.text.x = element_text(
        angle = 45,
        hjust = 1
      )
    )
  
  return(p)
})

names(boxplots) <- contrast_names
options(
  repr.plot.width = 12,
  repr.plot.height = 10
)
boxplots$Early_vs_Control
boxplots$Advanced_vs_Control
boxplots$Advanced_vs_Early

In [ ]:
library(pROC)

contrast_names <- names(DEPlist)

roc_results <- list()
roc_plots <- list()

for (contrast_name in contrast_names) {
  
  # 各contrastの上位3タンパク質
  top3_proteins <- rownames(
    DEPlist[[contrast_name]] %>%
      arrange(adj.P.Val) %>%
      head(3)
  )
  
  # ROCを計算
  roc_list <- lapply(top3_proteins, function(protein) {
    
    roc(
      response = pro_sample_val$CaCo_Status,
      predictor = pro_data_val[, protein]
    )
  })
  
  names(roc_list) <- top3_proteins
  
  roc_results[[contrast_name]] <- roc_list
  
  # ROC plot
  plot(
    roc_list[[1]],
    col = 1,
    lwd = 2,
    main = paste0("ROC Curve: ", contrast_name)
  )
  
  if (length(roc_list) >= 2) {
    for (i in 2:length(roc_list)) {
      plot(
        roc_list[[i]],
        col = i,
        lwd = 2,
        add = TRUE
      )
    }
  }
  
  # AUCをlegendに表示
  legend_text <- paste0(
    names(roc_list),
    " (AUC = ",
    round(sapply(roc_list, auc), 3),
    ")"
  )
  
  legend(
    "bottomright",
    legend = legend_text,
    col = seq_along(roc_list),
    lwd = 2,
    cex = 0.8
  )
  
  roc_plots[[contrast_name]] <- roc_list
}
options(
  repr.plot.width = 8,
  repr.plot.height = 4
)
roc_plots$Early_vs_Control
roc_plots$Advanced_vs_Control
roc_plots$Advanced_vs_Early

# Export data

In [ ]:
write.csv(target_proteins, file = "./Data/target_labels.csv", row.names = FALSE)

In [ ]:
write.csv(result_table, file = "./Data/result_labels.csv", row.names = TRUE)

# ↓↓↓部位別で発現量に差があるかを見る↓↓↓

# Design Matrix

In [ ]:
sample_group <- c()
sample_gender <- c()
sample_age <- c()
sample_bmi <- c()

for (sample in rownames(pro_data_val)) {
    # group <- pro_sample_val[rownames(pro_data_val) %in% pro_sample_val$SampleID,]$progress_group
    matched_row <- pro_sample_val[pro_sample_val$SampleID == sample, ]
    
    group <- if(nrow(matched_row) > 0) matched_row$progress_group else NA
    sample_group <- c(sample_group, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$sex else NA
    sample_gender <- c(sample_gender, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$fvage else NA
    sample_age <- c(sample_age, group)
    
    group <- if(nrow(matched_row) > 0) matched_row$bmi else NA
    sample_bmi <- c(sample_bmi, group)
    
    # print(sample_group)
    # table(rownames(pro_data_val) %in% pro_sample_val$SampleID)
}
group_order <- c(0, 1, 2)
sample_group <- factor(sample_group, levels = group_order, labels = c("Control", "Early_Stage", "Advanced_Stage"))
group_order <- c(1,2)
sample_gender <- factor(sample_gender, levels = group_order, labels = c("Male", "Female"))

design <- model.matrix(~0 + sample_group + sample_gender + sample_age + sample_bmi)
colnames(design) <- c("Control", "Early_Stage", "Advanced_Stage", "Female", "Age", "BMI")
design

## Check

In [ ]:
#check
tempD <- apply(design, 2, sum)
tempD <- tibble(Group = names(tempD), DesignMatrix = tempD)
tempD2 <- pro_sample_val
# print(tempD)
# str(tempD2$progress_group)

tempD2 %>%
    dplyr::mutate(Group = factor(progress_group, levels = c(0, 1, 2), labels = c("Control", "Early_Stage", "Advanced_Stage"))) %>%
    # dplyr::mutate(Group = as.character(progress_group)) %>%
    dplyr::group_by(Group) %>%
    dplyr::summarize(nSamples = n()) %>%
    dplyr::left_join(tempD, by = "Group")

## Result of Limma

In [ ]:
dim(t(pro_data_val))
dim(design)

In [ ]:
#Limmaでやってみる
library(limma)
pro_data_val_t <- t(pro_data_val)


fit <- lmFit(pro_data_val_t, design)
contrast <- makeContrasts(
    Early_vs_Control = Early_Stage - Control,
    Advanced_vs_Control = Advanced_Stage - Control,
    Advanced_vs_Early = Advanced_Stage - Early_Stage,
    levels = design
)
fit2 <- contrasts.fit(fit, contrast)
fit2 <- eBayes(fit2)

#Early_vs_Control
print("Early_vs_Control")
result_Early_vs_Control <- topTable(
    fit2,
    coef = "Early_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Early_vs_Control)

sig_raw <- sum(result_Early_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Early_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Early_vs_Control$adj.P.Val < 0.05 &
    abs(result_Early_vs_Control$logFC) > 0.5
)


#Advanced_vs_Control
print("Advanced_vs_Control")
result_Advanced_vs_Control <- topTable(
    fit2,
    coef = "Advanced_vs_Control",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Control)

sig_raw <- sum(result_Advanced_vs_Control$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Control$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Control$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Control$logFC) > 0.5
)

#Advanced_vs_Early
print("Advanced_vs_Early")
result_Advanced_vs_Early <- topTable(
    fit2,
    coef = "Advanced_vs_Early",
    number = Inf,
    adjust.method = "BH"
)

head(result_Advanced_vs_Early)

sig_raw <- sum(result_Advanced_vs_Early$P.Value < 0.05)
sig_adj <- sum(result_Advanced_vs_Early$adj.P.Val < 0.05)

cat("Number of p value < 0.05:", sig_raw, "\n")
cat("Number of Adjusted p value < 0.05:", sig_adj, "\n")

sum(
    result_Advanced_vs_Early$adj.P.Val < 0.05 &
    abs(result_Advanced_vs_Early$logFC) > 0.5
)